# 语言模型零基础 03：FSA、FST 与第一张 OpenFst 图

前两课把语言模型看成概率公式；这一课把它变成 **状态、弧、标签和权重**，并使用真实 OpenFst 命令编译、检查、绘图和寻找最短路径。

完成本课后，你应该能够：

1. 区分 FSA、FST 与 WFST；
2. 读懂 OpenFst 文本格式中的每一列；
3. 解释初始状态、终止状态、epsilon 和 symbol table；
4. 使用 `fstcompile`、`fstinfo`、`fstprint`、`fstdraw`、`fstshortestpath`；
5. 解释为什么最高概率路径对应最低 `-log` 代价。


## 运行环境

本机已在 Ubuntu WSL 中安装 OpenFst 命令行工具。Notebook 的 Python 内核运行在 Windows，固定命令通过 `wsl.exe` 调用 Linux 中的 OpenFst。

如果以后把课程复制到另一台 Windows 电脑，可以在安装好 Ubuntu WSL 后运行：

```powershell
wsl -d Ubuntu -u root -- apt-get install -y libfst-tools graphviz
```


In [ ]:
from pathlib import Path
from math import exp, log
import shlex
import subprocess

def find_project_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("请从 learn_asr 或 notebooks 目录启动 Jupyter")

ROOT = find_project_root()
LAB = ROOT / "openfst_lab" / "lesson03"
LAB.mkdir(parents=True, exist_ok=True)

def run_wsl(*args, check=True):
    result = subprocess.run(
        ["wsl", "-d", "Ubuntu", "--", *map(str, args)],
        text=True, capture_output=True, check=False, encoding="utf-8", errors="replace"
    )
    if check and result.returncode != 0:
        raise RuntimeError(f"命令失败：{args}\n{result.stderr}")
    return result

def to_wsl_path(path):
    """把 Windows 绝对路径稳定映射为 WSL /mnt/<drive>/... 路径。"""
    resolved = Path(path).resolve()
    drive = resolved.drive.rstrip(":").lower()
    relative = resolved.relative_to(resolved.anchor).as_posix()
    return f"/mnt/{drive}/{relative}"

print("项目根目录：", ROOT)
print("实验输出目录：", LAB)
print("fstcompile：", run_wsl("which", "fstcompile").stdout.strip())


## 1. FSA、FST、WFST 是什么？

| 名称 | 一条弧携带什么 | 主要作用 |
|---|---|---|
| FSA / acceptor | 一个输入标签 | 接受或拒绝一个符号序列 |
| FST / transducer | 输入标签和输出标签 | 把一种符号序列映射为另一种 |
| WFSA / WFST | 标签外再加权重 | 在接受或映射的同时给路径打分 |

例子：

- 语言模型 `G` 通常是加权 acceptor：输入和输出都是词，作用是限制并打分；
- 发音词典 `L` 是 transducer：输入音素或 token，输出词；
- ASR 解码是在许多路径中寻找总代价最低的路径。


## 2. OpenFst 文本格式

Acceptor 的弧：

```text
源状态  目标状态  标签  权重
```

Transducer 的弧：

```text
源状态  目标状态  输入标签  输出标签  权重
```

终止状态单独占一行：

```text
终止状态  可选终止权重
```

文本中第一条弧的源状态通常成为初始状态。OpenFst 的二进制 FST 才是后续算法真正读取的对象。


## 3. 构造一个 Bigram 风格的小语言模型

我们允许两条句子路径：

```text
wo ai xuexi
wo ai ASR
```

这里用拼音标签避免绘图字体问题。`xuexi` 和 `ASR` 的分支概率分别设为 0.8 和 0.2，因此对应代价是 `-log(0.8)` 和 `-log(0.2)`。


In [ ]:
p_xuexi = 0.8
p_asr = 0.2
cost_xuexi = -log(p_xuexi)
cost_asr = -log(p_asr)

print(f"xuexi: probability={p_xuexi:.1f}, cost={cost_xuexi:.6f}")
print(f"ASR:    probability={p_asr:.1f}, cost={cost_asr:.6f}")
print("概率较大的分支代价更小：", cost_xuexi < cost_asr)


In [ ]:
symbols_path = LAB / "words.syms"
text_path = LAB / "tiny_g.txt"
fst_path = LAB / "tiny_g.fst"

symbols_path.write_text(
    "<eps> 0\nwo 1\nai 2\nxuexi 3\nASR 4\n", encoding="utf-8", newline="\n"
)
text_path.write_text(
    "\n".join([
        "0 1 wo 0.0",
        "1 2 ai 0.0",
        f"2 3 xuexi {cost_xuexi:.9f}",
        f"2 3 ASR {cost_asr:.9f}",
        "3",
        "",
    ]),
    encoding="utf-8", newline="\n",
)

print("symbol table:\n" + symbols_path.read_text(encoding="utf-8"))
print("FST text:\n" + text_path.read_text(encoding="utf-8"))


### Symbol table 为什么必要？

OpenFst 内部标签是整数。Symbol table 负责在人类可读字符串和整数 ID 之间映射。这里 `<eps>` 必须使用 ID `0`。

常见错误：两张要组合的图看起来都有“同一个词”，但整数 ID 或 symbol table 不一致，导致 composition 得不到预期路径。

Windows 还要注意行尾：Linux OpenFst 可能把 CRLF 中残留的 `\r` 当作字段内容。本课写文本时显式使用 `newline=\"\n\"`，保证生成 LF 行尾。


In [ ]:
symbols_wsl = to_wsl_path(symbols_path)
text_wsl = to_wsl_path(text_path)
fst_wsl = to_wsl_path(fst_path)

compile_command = [
    "fstcompile",
    "--acceptor=true",
    f"--isymbols={symbols_wsl}",
    f"--osymbols={symbols_wsl}",
    "--keep_isymbols=true",
    "--keep_osymbols=true",
    text_wsl,
    fst_wsl,
]
print("执行：", " ".join(shlex.quote(x) for x in compile_command))
run_wsl(*compile_command)
print("已生成：", fst_path, "字节数：", fst_path.stat().st_size)


上一个单元格做了关键转换：

```text
人类可编辑文本 --fstcompile--> OpenFst 二进制图
```

以后 `fstcompose`、`fstdeterminize`、`fstminimize` 等算法处理的是右侧二进制图。


In [ ]:
info = run_wsl("fstinfo", fst_wsl).stdout
print(info)

for important_line in ["# of states", "# of arcs", "input deterministic", "acceptor"]:
    match = next((line for line in info.splitlines() if line.strip().startswith(important_line)), None)
    print("重点：", match)


`fstinfo` 是排错第一站。至少检查：状态数、弧数、是否 acceptor、是否 input deterministic、symbol table 是否存在，以及权重类型。


In [ ]:
printed = run_wsl(
    "fstprint",
    f"--isymbols={symbols_wsl}",
    f"--osymbols={symbols_wsl}",
    fst_wsl,
).stdout
print(printed)


`fstprint` 把二进制图重新打印为文本。即使原来按 acceptor 编译，打印时通常仍会看到 input/output 两列相同的标签；“两列相同”正是 acceptor 的特征。


In [ ]:
from IPython.display import SVG, display

dot_path = LAB / "tiny_g.dot"
svg_path = LAB / "tiny_g.svg"
run_wsl(
    "fstdraw",
    f"--isymbols={symbols_wsl}",
    f"--osymbols={symbols_wsl}",
    fst_wsl,
    to_wsl_path(dot_path),
)
run_wsl("dot", "-Tsvg", to_wsl_path(dot_path), "-o", to_wsl_path(svg_path))
display(SVG(filename=str(svg_path)))


读图时逐项确认：

- 状态 `0` 是初始状态；
- 双圆圈或指向终点的标记表示 final state；
- 每条弧包含输入标签、输出标签和权重；
- 两条路径共享 `wo ai` 前缀，然后在状态 `2` 分叉。


## 4. 最短路径就是最高概率路径

路径概率相乘：

$$P(path)=\prod_i p_i$$

使用 `cost_i=-log(p_i)` 后，路径代价相加：

$$cost(path)=\sum_i -\log(p_i)=-\log P(path)$$

所以最大概率路径等价于最小代价路径。


In [ ]:
best_path = LAB / "best.fst"
best_wsl = to_wsl_path(best_path)
run_wsl("fstshortestpath", fst_wsl, best_wsl)
best_text = run_wsl(
    "fstprint",
    f"--isymbols={symbols_wsl}",
    f"--osymbols={symbols_wsl}",
    best_wsl,
).stdout
print(best_text)
print("预期最佳路径包含 xuexi，因为 0.8 > 0.2。")


`fstshortestpath` 可能重新编号状态，`fstprint` 也不保证按路径遍历顺序排列各行。因此恢复 token 序列时必须先确定初始状态，再沿每条弧的 `source → destination` 走到 final state；不能直接从上到下读取标签。

### 滑块实验：什么时候最佳路径会切换？

拖动 `P(xuexi | wo ai)`。另一个分支概率自动使用 `1-p`。当两者越过 0.5 时，观察最短路径的变化。


In [ ]:
import ipywidgets as widgets

def labels_from_printed_path(text):
    """按状态连接关系遍历单路径 FST，不能假设 fstprint 行顺序就是路径顺序。"""
    arcs = {}
    destination_states = set()
    final_states = set()
    for line in text.splitlines():
        fields = line.split()
        if len(fields) >= 4:
            source, destination = int(fields[0]), int(fields[1])
            arcs[source] = (destination, fields[2])
            destination_states.add(destination)
        elif fields:
            final_states.add(int(fields[0]))
    start_candidates = set(arcs) - destination_states
    if len(start_candidates) != 1:
        raise ValueError(f"无法确定唯一初始状态：{start_candidates}")
    state = start_candidates.pop()
    labels = []
    visited = set()
    while state not in final_states:
        if state in visited or state not in arcs:
            raise ValueError("打印结果不是一条可完整遍历的无环路径")
        visited.add(state)
        state, label = arcs[state]
        if label != "<eps>":
            labels.append(label)
    return labels

def best_path_demo(p_xuexi=0.8):
    p_asr = 1.0 - p_xuexi
    text_path.write_text(
        "\n".join([
            "0 1 wo 0.0",
            "1 2 ai 0.0",
            f"2 3 xuexi {-log(p_xuexi):.9f}",
            f"2 3 ASR {-log(p_asr):.9f}",
            "3", "",
        ]), encoding="utf-8", newline="\n"
    )
    run_wsl(*compile_command)
    run_wsl("fstshortestpath", fst_wsl, best_wsl)
    output = run_wsl(
        "fstprint", f"--isymbols={symbols_wsl}", f"--osymbols={symbols_wsl}", best_wsl
    ).stdout
    labels = labels_from_printed_path(output)
    print(f"P(xuexi)={p_xuexi:.2f}, P(ASR)={p_asr:.2f}")
    print("最短路径：", " ".join(labels))
    return labels

widgets.interact(
    best_path_demo,
    p_xuexi=widgets.FloatSlider(value=0.8, min=0.05, max=0.95, step=0.05),
);

default_labels = best_path_demo(0.8)
assert default_labels == ["wo", "ai", "xuexi"], default_labels
print("默认路径顺序校验通过。")


## 5. FST：输入和输出可以不同

下面构造一个极小发音词典：

```text
jin tian  -> 今天
tian qi   -> 天气
```

第一段音节暂时输出 epsilon，读到完整词时才输出词。这里 epsilon 表示“不消费或不产生普通符号”，不是空格。


In [ ]:
phone_syms = LAB / "phones.syms"
word_syms = LAB / "lexicon_words.syms"
lexicon_txt = LAB / "lexicon.txt"
lexicon_fst = LAB / "lexicon.fst"

phone_syms.write_text("<eps> 0\njin 1\ntian 2\nqi 3\n", encoding="utf-8", newline="\n")
word_syms.write_text("<eps> 0\n今天 1\n天气 2\n", encoding="utf-8", newline="\n")
lexicon_txt.write_text(
    "0 1 jin <eps> 0\n1 2 tian 今天 0\n0 3 tian <eps> 0\n3 2 qi 天气 0\n2\n",
    encoding="utf-8", newline="\n",
)

run_wsl(
    "fstcompile",
    f"--isymbols={to_wsl_path(phone_syms)}",
    f"--osymbols={to_wsl_path(word_syms)}",
    "--keep_isymbols=true",
    "--keep_osymbols=true",
    to_wsl_path(lexicon_txt),
    to_wsl_path(lexicon_fst),
)
print(run_wsl(
    "fstprint",
    f"--isymbols={to_wsl_path(phone_syms)}",
    f"--osymbols={to_wsl_path(word_syms)}",
    to_wsl_path(lexicon_fst),
).stdout)
print("acceptor?", next(line for line in run_wsl('fstinfo', to_wsl_path(lexicon_fst)).stdout.splitlines() if line.strip().startswith('acceptor')))


语言模型 `G` 是 acceptor，而发音词典 `L` 是 transducer。以后组合 `L ∘ G` 时，核心条件是：

```text
L 的 output label 空间 == G 的 input label 空间
```

只看字符串名称还不够，还要检查整数 ID 和 symbol table。


## 6. Tropical 与 Log semiring 的最低限度理解

OpenFst 不只保存数字，还规定权重如何“相加”和“相乘”，这套规则叫 semiring。现在只记住：

- 常见 `standard` arc 使用 tropical semiring：路径内权重普通相加，多条竞争路径取最小值；
- log semiring 在合并多条路径时保留概率求和的含义；
- 寻找单条最佳路径时，负对数代价和 tropical 的 `min +` 直觉最直接。

不要把“概率相加”“路径代价相加”和 semiring 的 `Plus/Times` 名称混为一谈。后面做 determinization 时再深入。


## 7. 自动判题


In [ ]:
# 请修改五个答案
answer_1 = ""  # 一条弧同时有不同 input/output label：FSA 还是 FST？
answer_2 = None  # <eps> 在 symbol table 中通常使用哪个整数 ID？
answer_3 = ""  # 使用 -log 概率后，最佳路径是 min cost 还是 max cost？填 min/max
answer_4 = ""  # 把文本图编译为二进制图的命令名
answer_5 = ""  # L 的 output labels 必须与 G 的哪一侧 labels 匹配？填 input/output

checks = [
    str(answer_1).strip().upper() == "FST",
    answer_2 is not None and int(answer_2) == 0,
    str(answer_3).strip().lower() == "min",
    str(answer_4).strip().lower() == "fstcompile",
    str(answer_5).strip().lower() == "input",
]
for number, ok in enumerate(checks, 1):
    print(("✅" if ok else "❌"), f"第 {number} 题")
print(f"得分：{sum(checks)}/5")
if all(checks):
    print("通过：你已经能读写并运行第一张真实 OpenFst 图。")
else:
    print("回到对应实验，用 fstprint/fstinfo 找证据后再答。")


<details><summary>完成后展开参考答案</summary>

1. `FST`；2. `0`；3. `min`；4. `fstcompile`；5. `input`。

</details>

## 离场票

不看上文，画一张能接受 `wo ai xuexi` 和 `wo ai ASR` 的图，标出初始状态、终止状态、标签、权重，并写出对应 OpenFst 文本。然后用一句话分别解释：`fstcompile`、`fstprint`、`fstinfo` 和 `fstshortestpath`。

下一课：打开 `语言模型零基础_04_OpenFst组合确定化与最小化.ipynb`，真实执行 `compose`、`project`、`rmepsilon`、`determinize`、`minimize`、`arcsort` 和等价性检查。
